In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data, ClusterData, ClusterLoader
from torch_geometric.nn import SAGEConv

In [3]:
class NormedLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(NormedLinear, self).__init__()
        self.weight = nn.Parameter(torch.Tensor(in_features, out_features))
        self.weight.data.uniform_(-1, 1).renorm_(2, 1, 1e-5).mul_(1e5)
    def forward(self, x):
        out = F.normalize(x, dim=1).mm(F.normalize(self.weight, dim=0))
        return 10 * out

class Encoder(nn.Module):
    """GraphSAGE Encoder"""
    def __init__(self, x_dim, num_cls, hid_dim=128):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(x_dim, hid_dim)
        self.conv = SAGEConv(hid_dim, hid_dim)
        self.relu = nn.ReLU()
        self.classifier = NormedLinear(hid_dim, num_cls)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.relu(self.fc1(x))
        x = self.relu(self.conv(x, edge_index))
        out = self.classifier(x)
        return out

In [4]:
import anndata
adata = anndata.read_h5ad("/mnt/jwh83-data/Confetti/output/Redsea/Clustering/cellpose_cyto3_beforeREDSEA_leiden.h5ad")
adata

/home/labuser/anaconda3/envs/ns_env/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 145925 × 45
    obs: 'CD123', 'CDX2', 'slide_name', 'CellID', 'Unnamed: 0', 'cell_size', 'x_centroid', 'y_centroid', 'FILE', 'MUC6', 'GATA3', 'Lefty', 'CD279', 'NKG2D', 'CK7', 'PGP95', 'CD154', 'Somatostatin', 'CD294', 'DRAQ5', 'Hoechst1', 'leiden'
    uns: 'leiden', 'neighbors', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [5]:
def adata_to_dataframe(adata):
    """
    Convert AnnData object to DataFrame.

    Parameters:
        adata (AnnData): AnnData object to be converted.

    Returns:
        DataFrame: DataFrame containing both variables and observation attributes.
    """
    # Convert AnnData variables to DataFrame
    df_var = adata.to_df()

    # Extract observation attributes from AnnData
    df_obs = adata.obs

    # Concatenate variable DataFrame and observation attribute DataFrame along axis 1
    df = pd.concat([df_var, df_obs], axis=1)

    return df

In [6]:
adata_df = adata_to_dataframe(adata)
adata_df

,CD45RO,CD56,CD15,CD163,MUC2,CD34,CD36,aDefensin5,CD49a,HLA-DR,...,CD279,NKG2D,CK7,PGP95,CD154,Somatostatin,CD294,DRAQ5,Hoechst1,leiden
0,-0.866286,-0.402901,-0.405809,-0.525689,-0.130551,-0.469013,-0.356011,-0.180977,-0.909609,-0.652103,...,-0.539121,-0.291249,-0.276408,-0.366170,-0.226150,-0.383719,-0.341282,0.000000,3709.505495,0
1,-0.810474,-0.381786,-0.405809,-0.521973,0.074439,-0.457526,-0.354982,-0.180957,-0.207217,-0.602877,...,-0.517543,-0.280265,-0.276330,-0.347321,-0.225558,-0.372722,-0.337252,24.422261,2364.395760,29
2,-0.770204,-0.375506,-0.405809,-0.501351,0.041411,-0.411419,-0.341711,-0.180693,0.183106,-0.577302,...,-0.510007,-0.275242,-0.275234,-0.333747,-0.223802,-0.351947,-0.333972,134.000000,2663.953819,29
3,-0.723534,-0.247989,-0.405809,-0.489288,-0.148140,0.127775,-0.156897,-0.179703,0.762135,-0.523340,...,-0.476669,-0.251027,-0.268149,-0.291584,-0.217332,-0.348816,-0.325299,163.549367,2126.275949,7
4,-0.760077,-0.386900,-0.405809,-0.517853,-0.197608,-0.438094,-0.354281,-0.180750,0.323887,-0.597468,...,-0.509373,-0.271757,-0.275457,-0.333802,-0.223827,-0.365327,-0.332819,147.184211,2316.745614,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145920,-0.226018,-0.398738,-0.405809,-0.506632,-0.302002,-0.296718,-0.331389,-0.172022,-0.825901,-0.541391,...,-0.108597,-0.240311,-0.241084,-0.311135,-0.212813,-0.251273,-0.277713,1770.360507,5836.139493,22
145921,-0.337071,-0.377177,-0.404732,-0.509969,-0.334164,-0.331465,-0.332713,-0.171560,-0.845097,-0.537359,...,-0.003954,-0.248717,-0.165476,-0.307447,-0.215247,-0.258779,-0.286079,1616.012500,6101.376389,22
145922,-0.119838,-0.398535,-0.391066,-0.508455,-0.311397,-0.293502,-0.338064,-0.174852,-0.810197,-0.544713,...,-0.086873,-0.209757,-0.239219,-0.312460,-0.209678,-0.265442,-0.282956,1730.558020,5893.924915,25
145923,-0.217355,-0.395871,-0.405809,-0.506699,-0.334418,-0.296287,-0.264409,-0.157786,-0.849436,-0.490612,...,0.041564,-0.239016,-0.178388,-0.301613,-0.212123,-0.273700,-0.283799,138.468490,746.450654,22


In [7]:
adata_df.columns

Index(['CD45RO', 'CD56', 'CD15', 'CD163', 'MUC2', 'CD34', 'CD36', 'aDefensin5',
       'CD49a', 'HLA-DR', 'CD38', 'CollagenIV', 'CD4', 'CD138', 'CD44',
       'Vimentin', 'CD66', 'Podoplanin', 'CHGA', 'CD3', 'SOX9', 'CD161',
       'Synaptophysin', 'CD57', 'ITLN1', 'CD127', 'CD45', 'CD49f', 'aSMA',
       'Cytokeratin', 'CD8', 'CD19', 'BCL2', 'CD90', 'CD21', 'CD7', 'CD11c',
       'Ki67', 'CD68', 'MUC1', 'CD206', 'CD16', 'CD117', 'CD69', 'CD31',
       'CD123', 'CDX2', 'slide_name', 'CellID', 'Unnamed: 0', 'cell_size',
       'x_centroid', 'y_centroid', 'FILE', 'MUC6', 'GATA3', 'Lefty', 'CD279',
       'NKG2D', 'CK7', 'PGP95', 'CD154', 'Somatostatin', 'CD294', 'DRAQ5',
       'Hoechst1', 'leiden'],
      dtype='object')

In [8]:
adata_df.columns.get_loc('CD123')

45

In [9]:
# create combined region identifier
#adata_df["unique_region"] = (
#    adata_df["sample_ID"].astype(str) + "_" + adata_df["segmentation_method"].astype(str)
#)

In [10]:
adata_df.to_csv("/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260225_ns444_cellpose_cyto3_beforeREDSEA_leiden.csv", index=False)

In [11]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix

# -------------------------------------------------------
# 0) Paths + config (EDIT)
# -------------------------------------------------------
TRAIN_CSV = "/mnt/jwh83-data/Confetti/output/Redsea/STELLAR/20251007_cleaned_trainingdata_yang.csv"  # used to reconstruct label_map + feature cols
MODEL_PT  = "/mnt/jwh83-data/Confetti/output/Redsea/STELLAR/fcnet_minigraphs_region_split_hubmapallregion.pt"

NEW_CSV   = "/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260225_ns444_cellpose_cyto3_beforeREDSEA_leiden.csv"
OUT_PREDS = "/mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260225_ns444_cellpose_cyto3_beforeREDSEA_gnn_predict.csv"

# graph build params (MUST match your preprocessing)
marker_cols    = 45
coord_cols     = ('x_centroid', 'y_centroid')
label_col      = "cell_type_update"
region_col     = 'slide_name'
distance_thres = 100
cluster_size   = 300
sample_rate    = 1.0
seed           = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------
# 1) Reconstruct training artifacts (feature cols + label_map)
#    IMPORTANT: label_map order must match training
# -------------------------------------------------------
train_df = pd.read_csv(TRAIN_CSV)

# features are first marker_cols columns (same as your build_mini_graphs)
feature_cols = train_df.columns[:marker_cols].tolist()

y_str_train = train_df[label_col].astype(str).values if label_col in train_df.columns else train_df.iloc[:, 0].astype(str).values
label_names = np.sort(np.unique(y_str_train))
label_map = {n: i for i, n in enumerate(label_names)}
num_cls = len(label_map)
in_dim = len(feature_cols)

print(f"[Training artifacts] in_dim={in_dim}, num_cls={num_cls}")

# -------------------------------------------------------
# 2) Build mini-graphs for NEW dataset, BUT keep mapping to original row indices
#    (This is a graph-builder variant for inference.)
# -------------------------------------------------------
from sklearn.metrics import pairwise_distances
from torch_geometric.data import Data

def build_mini_graphs_for_inference(
    csv_path,
    feature_cols,
    marker_cols=44,
    coord_cols=('x', 'y'),
    region_col=None,
    distance_thres=100.0,
    cluster_size=300,
    sample_rate=1.0,
    random_state=42,
):
    df = pd.read_csv(csv_path)
    print(f"[NEW] Total cells: {len(df)}")

    # keep original row id for writing predictions back
    df["_row_id_"] = np.arange(len(df), dtype=np.int64)

    # optional sampling (keeps subset only)
    if sample_rate < 1.0:
        df = df.sample(frac=sample_rate, random_state=random_state).reset_index(drop=True)
        print(f"[NEW] Sampled {len(df)} cells ({sample_rate*100:.1f}%)")

    # ensure all feature cols exist; fill missing with 0.0
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0

    X_all = df[feature_cols].to_numpy(dtype=np.float32)
    pos_all = df.loc[:, list(coord_cols)].to_numpy(dtype=np.float32)
    row_ids = df["_row_id_"].to_numpy(dtype=np.int64)

    if region_col is None:
        df["_REGION_"] = "ALL"
        region_col_use = "_REGION_"
    else:
        if region_col not in df.columns:
            raise ValueError(f"region_col='{region_col}' not found in NEW CSV columns.")
        region_col_use = region_col

    regions = df[region_col_use].astype(str).values
    unique_regions = np.unique(regions)
    rng = np.random.RandomState(random_state)

    graphs = []
    for reg in unique_regions:
        reg_mask = (regions == reg)
        reg_idx = np.where(reg_mask)[0]
        if len(reg_idx) == 0:
            continue

        reg_idx = reg_idx.copy()
        rng.shuffle(reg_idx)

        num_clusters = int(np.ceil(len(reg_idx) / cluster_size))
        for i in range(num_clusters):
            sub_local = reg_idx[i*cluster_size : (i+1)*cluster_size]
            if len(sub_local) == 0:
                continue

            X_sub = X_all[sub_local]
            pos_sub = pos_all[sub_local]
            row_sub = row_ids[sub_local]   # mapping to original NEW df row indices

            dists = pairwise_distances(pos_sub)
            dmask = dists < distance_thres
            np.fill_diagonal(dmask, 0)

            edges = np.transpose(np.nonzero(dmask))
            edge_index = (
                torch.LongTensor(edges).T
                if len(edges) > 0
                else torch.empty((2, 0), dtype=torch.long)
            )

            g = Data(
                x=torch.FloatTensor(X_sub),
                edge_index=edge_index,
            )
            g.region = reg
            g.row_id = torch.LongTensor(row_sub)   # <-- critical for stitching predictions back
            graphs.append(g)

    print(f"[NEW] Built {len(graphs)} mini-graphs across {len(unique_regions)} region(s)")
    return df, graphs

new_df_full, graphs_new = build_mini_graphs_for_inference(
    NEW_CSV,
    feature_cols=feature_cols,
    marker_cols=marker_cols,
    coord_cols=coord_cols,
    region_col=region_col,
    distance_thres=distance_thres,
    cluster_size=cluster_size,
    sample_rate=sample_rate,
    random_state=seed,
)

# -------------------------------------------------------
# 3) Load model weights (same Encoder as training)
# -------------------------------------------------------
# You must have Encoder defined exactly as in training.
# Example:
# model = Encoder(in_dim, num_cls).to(device)

model = Encoder(in_dim, num_cls).to(device)

ckpt = torch.load(MODEL_PT, map_location=device)
# GraphBatchTrainer saved {"model_state": ...}
model.load_state_dict(ckpt["model_state"])
model.eval()

print("[Model] Loaded weights and set to eval()")

# -------------------------------------------------------
# 4) Inference: predict node labels for every graph, stitch back by row_id
# -------------------------------------------------------
N_total = len(pd.read_csv(NEW_CSV))
pred_id = np.full(N_total, -1, dtype=np.int64)
pred_conf = np.full(N_total, np.nan, dtype=np.float32)

with torch.no_grad():
    for g in graphs_new:
        g = g.to(device)
        logits = model(g)                    # (n_nodes, C)
        probs = F.softmax(logits, dim=1)
        ids = probs.argmax(dim=1).cpu().numpy()
        conf = probs.max(dim=1).values.cpu().numpy()

        rows = g.row_id.cpu().numpy()
        pred_id[rows] = ids
        pred_conf[rows] = conf

pred_name = np.array([label_names[i] if i >= 0 else "NA" for i in pred_id], dtype=object)

# -------------------------------------------------------
# 5) Save CSV with predictions
# -------------------------------------------------------
out_df = pd.read_csv(NEW_CSV)
out_df["pred_id"] = pred_id
out_df["pred_name"] = pred_name
out_df["pred_conf"] = pred_conf

os.makedirs(os.path.dirname(OUT_PREDS) or ".", exist_ok=True)
out_df.to_csv(OUT_PREDS, index=False)
print(f"Saved: {OUT_PREDS}")

# -------------------------------------------------------
# 6) Optional evaluation if NEW_CSV has ground truth labels
# -------------------------------------------------------
if label_col is not None and label_col in out_df.columns:
    y_true_str = out_df[label_col].astype(str).values
    # map unseen labels to -1
    y_true = np.array([label_map.get(v, -1) for v in y_true_str], dtype=np.int64)

    valid = (y_true >= 0) & (pred_id >= 0)
    if valid.sum() > 0:
        acc = accuracy_score(y_true[valid], pred_id[valid])
        print(f"Eval on {valid.sum()} labeled rows: accuracy={acc:.4f}")

        cm = confusion_matrix(y_true[valid], pred_id[valid], labels=np.arange(num_cls))
        print("Confusion matrix shape:", cm.shape)
    else:
        print("No valid labeled rows for evaluation (labels missing or unseen).")


[Training artifacts] in_dim=45, num_cls=28
[NEW] Total cells: 145925
[NEW] Built 491 mini-graphs across 8 region(s)
[Model] Loaded weights and set to eval()
Saved: /mnt/jwh83-data/Confetti/output/Redsea/Clustering/20260225_ns444_cellpose_cyto3_beforeREDSEA_gnn_predict.csv
